In [1]:
import os
import pandas as pd
from Bio import Entrez
from dotenv import load_dotenv

In [2]:
print(os.getcwd())

load_dotenv()

email_address = os.getenv("EMAIL_ADDRESS")
Entrez.email = email_address

/Users/wes/Desktop/MSc-Data-Science/MSc-Data-Science/MAST7865 - Data Science Project/biomedical_kg_thesis/exploration


In [3]:
def fetch_papers(query, engine, num_articles):
    search_handle = Entrez.esearch(
        db=engine,
        term=query,
        retmax=num_articles
    )

    search_results = Entrez.read(search_handle)

    ids = search_results["IdList"]

    fetch_handle = Entrez.efetch(
        db=engine,
        id=",".join(ids),
        rettype="abstract",
        retmode="xml"
    )

    papers = Entrez.read(fetch_handle)

    return papers

In [4]:
def make_query_dict(query, engine, num_articles):

    papers = fetch_papers(query, engine, num_articles)

    query_dict = {
        "query text": query,
        "papers": papers
    }

    return query_dict

In [5]:
base_queries = [
    '"atrial fibrillation"[Title/Abstract]',
    '"heart failure"[Title/Abstract]',
    '"coronary artery disease"[Title/Abstract]',
    '"myocardial infarction"[Title/Abstract]',
    'hypertension[Title/Abstract]',
    'cardiomyopathy[Title/Abstract]',
    '"heart valve disease"[Title/Abstract]',
]

years = range(2020, 2027)

queries = [
    f'{q} AND "{year}"[Date - Publication]'
    for q in base_queries
    for year in years
]

Inspect `queries` dictionary

In [6]:
display(queries[0:3])
display(queries[-4:-1])

['"atrial fibrillation"[Title/Abstract] AND "2020"[Date - Publication]',
 '"atrial fibrillation"[Title/Abstract] AND "2021"[Date - Publication]',
 '"atrial fibrillation"[Title/Abstract] AND "2022"[Date - Publication]']

['"heart valve disease"[Title/Abstract] AND "2023"[Date - Publication]',
 '"heart valve disease"[Title/Abstract] AND "2024"[Date - Publication]',
 '"heart valve disease"[Title/Abstract] AND "2025"[Date - Publication]']

In [15]:
query_dicts = []

for query in queries:
    query_dicts.append(
        make_query_dict(
            query=query,
            engine="pubmed",
            num_articles=25
        )
    )

In [16]:
def extract_doi(article):
    for eid in article.get("ELocationID", []):
        if eid.attributes.get("EIdType") == "doi":
            return str(eid)
    return None

In [17]:
def construct_corpus(queries, summary: bool):

    """

    """

    # PMIDs
    pmids = []
    titles = []
    abstracts = []
    authors = []
    journals = []
    years = []
    dois = []
    search_queries = []

    for query in queries:

        papers = query["papers"]["PubmedArticle"]

        query_text = query["query text"]

        for paper in papers:

            medlinecitation = paper["MedlineCitation"]

            article = medlinecitation["Article"]

            # paper_id
            if medlinecitation.get("PMID"):
                paper_id = medlinecitation["PMID"]
            else:
                paper_id = None

            pmids.append(paper_id)

            # title
            if article.get("ArticleTitle"):
                title = f"{article ["ArticleTitle"]}"
            else:
                title = None

            titles.append(title)

            # abstract
            if article .get("Abstract"):
                if article ["Abstract"].get("AbstractText"):
                    abstract = " ".join(
                        str(text) for text in article ["Abstract"]["AbstractText"]
                    )
                else:
                    abstract = None
            else: abstract = None

            abstracts.append(abstract)

            # journal
            if article ["Journal"].get("Title"):
                journal = article ["Journal"]["Title"]
            else:
                journal = None

            journals.append(journal)

            # year
            if article["Journal"]["JournalIssue"]["PubDate"] .get("Year"):
                year = article["Journal"]["JournalIssue"]["PubDate"]["Year"]
            else:
                year = None

            years.append(year)

    corpus = pd.DataFrame({
        "pmid": pmids,
        "title": titles,
        "abstract": abstracts,
        "journal": journals,
        "year": years
    })

    processed_corpus = (
        corpus
        .drop_duplicates(subset="pmid")
        .dropna(subset=["pmid", "abstract"])
    )

    pre_length = len(corpus)
    post_length = len(processed_corpus)

    num_duplicates_dropped = len(corpus) - len(corpus.drop_duplicates(subset="pmid"))
    num_absna_dropped = len(corpus) - len(corpus.dropna(subset=["pmid", "abstract"]))

    if summary == True:
        construction_summary = pd.DataFrame({
            "Summary Metric": ["Value"],
            "Retrieved Papers": [pre_length],
            "Duplicates Removed": [num_duplicates_dropped],
            "Missing Abstracts Removed": [num_absna_dropped],
            "Final Corpus Size": [len(processed_corpus)]
        })

        construction_summary.set_index("Summary Metric", inplace=True)

        display(construction_summary.T)

    return processed_corpus

In [18]:
corpus = construct_corpus(query_dicts, summary=True)

Summary Metric,Value
Retrieved Papers,1209
Duplicates Removed,140
Missing Abstracts Removed,86
Final Corpus Size,987


In [19]:
corpus.set_index("pmid", inplace=True)
corpus.head()

,title,abstract,journal,year
pmid,,,,
34950328,Recurrent Takotsubo Cardiomyopathy During Cryo...,We report a case of 72-year-old female with pr...,Journal of atrial fibrillation,2020
34950327,Contemporary Anticoagulation Practices for Pos...,Postoperative atrial fibrillation (POAF) is a ...,Journal of atrial fibrillation,2020
34950325,Comparison of Fragmented Electrogram Based Str...,Ganglionated plexus (GP) ablation is an emergi...,Journal of atrial fibrillation,2020
34950322,Incidence of Early Atrial Fibrillation After T...,Post-operative atrial fibrillation (POAF) is c...,Journal of atrial fibrillation,2020
34950321,Effect of Intensive Blood Pressure Lowering on...,The effect of intensive versus standard blood ...,Journal of atrial fibrillation,2020


In [20]:
corpus.to_csv("../data/processed/contemporary/contemporary_corpus.csv", index=True)